# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamidism/Machine-Learning-intern/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Refresh / Content Opportunity Scoring**

I'm choosing this lane because I've already run the starter pipeline end-to-end in notebooks 01
and 02, and I saw firsthand how a learned model beats a hand-written rule at ranking pages by
refresh priority (Precision@50 jumped from 0.240 with the baseline rule to 0.740 with a random
forest). This gave me a working intuition for the workflow — problem framing, baseline, model,
evaluation — and I want to build on that foundation rather than start a new problem type from
scratch. It also matches a real, understandable business need: with limited review capacity,
knowing which pages to look at first has clear, practical value.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**Decision this improves:** Which pages a content reviewer should prioritize for refresh,
expansion, or protection, given limited review capacity.

**Who acts on it:** A content/SEO reviewer with time to review only a limited number of pages
per cycle (e.g., the top 20-50 candidates).

**Action taken:** The reviewer opens the top-ranked pages first and decides whether to refresh
the content, expand it, protect it from further decline, prune it, or simply monitor it — based
on the reason codes attached to each page's score.

**Cost of a wrong call:**
- False positive (flagged as urgent but wasn't): wastes reviewer time on a page that didn't
  need attention — a low-severity cost.
- False negative (a genuinely declining/valuable page never gets flagged): the page keeps
  losing visibility unnoticed, which is a higher-severity, silent cost since no one is alerted
  to intervene.

This asymmetry means the ranking should favor recall on high-impressions pages, even at the
cost of a few false positives further down the list.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Using the starter dataset (`data/raw/content_refresh_anonymized.csv`) to pull 2-3 numbers
that show why this lane is worth pursuing.

In [7]:
import pandas as pd

# Read directly from your GitHub repo's raw file URL
df = pd.read_csv("https://raw.githubusercontent.com/hamidism/Machine-Learning-intern/main/data/raw/content_refresh_anonymized.csv")

total_pages = len(df)
declining = (df["trend_direction"] == "down").sum()
pct_declining = declining / total_pages * 100

stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
pct_stale_visible = stale_visible / total_pages * 100

low_ctr_visible = ((df["impressions_90d"] >= 500) & (df["avg_position"] > 0) &
                    (df["avg_position"] <= 20) & (df["ctr"] < 0.5)).sum()

print(f"Total pages in starter dataset: {total_pages}")
print(f"Pages currently declining: {declining} ({pct_declining:.1f}%)")
print(f"Stale but still visible pages (180+ days old, 500+ impressions): {stale_visible} ({pct_stale_visible:.1f}%)")
print(f"Visible pages with weak CTR for their position: {low_ctr_visible}")

Total pages in starter dataset: 30000
Pages currently declining: 16262 (54.2%)
Stale but still visible pages (180+ days old, 500+ impressions): 17 (0.1%)
Visible pages with weak CTR for their position: 9759


On the 30,000-row starter dataset:
- 16,262 pages (54.2%) are currently flagged as declining — a large enough pool that prioritization
  genuinely matters; a reviewer cannot manually check all of them.
- 9,759 pages have weak CTR relative to their position tier — a strong signal that ranking by
  opportunity (not just declining status) adds real value beyond a single "is it declining" flag.
- Only 17 pages (0.1%) are both stale AND highly visible (500+ impressions) — showing that the
  most obvious rule-based candidates are rare, and a learned model is needed to surface less
  obvious but still valuable review candidates.

Additionally, the starter pipeline's own verified results (from `outputs/model_results.json`)
show the learned model substantially outperforms the hand-written baseline rule:

| Method | Precision@50 |
|---|---|
| Baseline rules | 0.240 |
| Random forest | 0.740 |

This ~3x lift, combined with the fact that over half the dataset is flagged as declining,
confirms there's both enough signal and enough scale in this problem to be worth the next
7 weeks of work.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.